# 📐 Statistiques & Échantillonnage
**Cours couverts :** Introduction to Statistics, Introduction to Statistics in Python, Sampling in Python

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)

# Population : salaires de 10 000 employés
population_salaires = np.random.normal(loc=50000, scale=12000, size=10000)
population_salaires = np.clip(population_salaires, 20000, 120000)

print(f"Population : N={len(population_salaires)}, μ={population_salaires.mean():.0f}, σ={population_salaires.std():.0f}")

## 1. Statistiques descriptives

In [ ]:
# Mesures de tendance centrale
data = population_salaires[:100]

print("=== TENDANCE CENTRALE ===")
print(f"Moyenne   : {np.mean(data):.1f}")
print(f"Médiane   : {np.median(data):.1f}")
print(f"Mode (app): {stats.mode(data.round(-3)).mode:.0f}")  # arrondi à millier

print("\n=== DISPERSION ===")
print(f"Variance  : {np.var(data):.1f}")
print(f"Ecart-type: {np.std(data):.1f}")
print(f"Etendue   : {np.ptp(data):.1f}")  # max - min
print(f"Q1        : {np.percentile(data, 25):.1f}")
print(f"Q3        : {np.percentile(data, 75):.1f}")
print(f"IQR       : {stats.iqr(data):.1f}")  # Q3 - Q1

print("\n=== FORME ===")
print(f"Asymétrie (skewness): {stats.skew(data):.3f}")
print(f"  > 0 : queue à droite, < 0 : queue à gauche")
print(f"Kurtosis            : {stats.kurtosis(data):.3f}")
print(f"  > 0 : queues plus épaisses que normale (leptokurtique)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution avec annotations
axes[0].hist(data, bins=20, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(np.mean(data), color="red", linestyle="--", label=f"Moyenne = {np.mean(data):.0f}")
axes[0].axvline(np.median(data), color="orange", linestyle="-", label=f"Médiane = {np.median(data):.0f}")
axes[0].set_title("Distribution des salaires")
axes[0].legend()

# Boxplot annoté
axes[1].boxplot(data, vert=True)
q1, median, q3 = np.percentile(data, [25, 50, 75])
iqr = q3 - q1
axes[1].annotate(f"Q3={q3:.0f}", xy=(1.1, q3), fontsize=9)
axes[1].annotate(f"Med={median:.0f}", xy=(1.1, median), fontsize=9)
axes[1].annotate(f"Q1={q1:.0f}", xy=(1.1, q1), fontsize=9)
axes[1].annotate(f"IQR={iqr:.0f}", xy=(0.6, (q1+q3)/2), fontsize=9)
axes[1].set_title("Boxplot annoté")

plt.tight_layout()
plt.show()

## 2. Distributions de probabilité

In [ ]:
x = np.linspace(-4, 4, 200)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Normale
axes[0, 0].plot(x, stats.norm.pdf(x), color="steelblue", linewidth=2)
axes[0, 0].fill_between(x, stats.norm.pdf(x), where=(x > -1) & (x < 1), alpha=0.3)
axes[0, 0].set_title("Normale N(0,1)\n~68% dans ±1σ")

# Variations de la normale
for mu, sigma, label in [(0, 1, "N(0,1)"), (0, 2, "N(0,2)"), (2, 1, "N(2,1)")]:
    axes[0, 1].plot(x, stats.norm.pdf(x, mu, sigma), label=label, linewidth=2)
axes[0, 1].set_title("Variations de la loi normale")
axes[0, 1].legend()

# t de Student
for df_val, label in [(1, "df=1"), (5, "df=5"), (30, "df=30"), (1000, "Normale")]:
    axes[0, 2].plot(x, stats.t.pdf(x, df_val), label=label, linewidth=2)
axes[0, 2].set_title("Distribution t de Student\n(queues plus épaisses pour petits df)")
axes[0, 2].legend()

# Binomiale
n_binom, p_binom = 20, 0.3
k = np.arange(0, n_binom+1)
axes[1, 0].bar(k, stats.binom.pmf(k, n_binom, p_binom), color="coral", edgecolor="white")
axes[1, 0].set_title(f"Binomiale B({n_binom},{p_binom})\nProbabilité de k succès")

# Poisson
lam = 5
k_poisson = np.arange(0, 20)
axes[1, 1].bar(k_poisson, stats.poisson.pmf(k_poisson, lam), color="mediumseagreen", edgecolor="white")
axes[1, 1].set_title(f"Poisson(λ={lam})\nNb événements rares")

# Exponentielle
x_exp = np.linspace(0, 5, 200)
for scale, label in [(0.5, "λ=2"), (1.0, "λ=1"), (2.0, "λ=0.5")]:
    axes[1, 2].plot(x_exp, stats.expon.pdf(x_exp, scale=scale), label=label, linewidth=2)
axes[1, 2].set_title("Exponentielle (temps entre événements)")
axes[1, 2].legend()

plt.suptitle("Distributions de probabilité", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Échantillonnage

In [ ]:
# Types d'échantillonnage

population_df = pd.DataFrame({
    "salaire": population_salaires,
    "departement": np.random.choice(["IT", "Marketing", "RH", "Finance"], 10000,
                                     p=[0.3, 0.25, 0.2, 0.25])
})

# 1. Échantillonnage aléatoire simple
echantillon_simple = population_df.sample(n=100, random_state=42)
print("Simple aléatoire - Moyenne:", echantillon_simple["salaire"].mean().round(0))

# 2. Échantillonnage stratifié (proportionnel)
echantillon_strat = population_df.groupby("departement", group_keys=False).apply(
    lambda x: x.sample(frac=0.01, random_state=42)
)
print("Stratifié - Distribution:", echantillon_strat["departement"].value_counts().to_dict())

# 3. Échantillonnage systématique (ex: prendre 1 sur 100)
pas = 100
debut = np.random.randint(0, pas)
echantillon_syst = population_df.iloc[debut::pas]
print(f"Systématique - N={len(echantillon_syst)}, Moyenne: {echantillon_syst['salaire'].mean():.0f}")

## 4. Théorème Central Limite (TCL)

In [ ]:
# TCL : la moyenne d'échantillons suit une loi normale,
# quelle que soit la distribution de la population
# → plus n est grand, plus c'est précis

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Population NON-normale (exponentielle)
pop_expo = np.random.exponential(scale=1, size=100000)

for idx, n_samp in enumerate([1, 5, 30]):
    # Tirer 10000 échantillons de taille n et calculer leur moyenne
    moyennes = [np.mean(np.random.choice(pop_expo, n_samp)) for _ in range(5000)]

    axes[0, idx].hist(pop_expo[:2000], bins=50, color="coral", alpha=0.7)
    axes[0, idx].set_title(f"Population (exponentielle)")
    axes[0, idx].set_xlim(0, 5)

    axes[1, idx].hist(moyennes, bins=50, color="steelblue", alpha=0.8)
    axes[1, idx].set_title(f"Dist. des moyennes (n={n_samp})\nσ_x̄ = {np.std(moyennes):.3f}")

plt.suptitle("Théorème Central Limite : plus n est grand, plus la distribution des moyennes est normale",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print("\nErreur standard théorique = σ/√n")
sigma = np.std(pop_expo)
for n in [1, 5, 30]:
    print(f"  n={n:2d} → σ/√n = {sigma/np.sqrt(n):.3f}")

## 5. Intervalles de confiance

In [ ]:
# Intervalle de confiance à 95% pour la moyenne
echantillon = population_salaires[:50]
n = len(echantillon)
x_bar = np.mean(echantillon)
s = np.std(echantillon, ddof=1)  # écart-type de l'échantillon

# Méthode 1 : formule manuelle (loi t)
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha/2, df=n-1)
erreur_std = s / np.sqrt(n)
ic_bas = x_bar - t_crit * erreur_std
ic_haut = x_bar + t_crit * erreur_std

print(f"Taille échantillon     : n = {n}")
print(f"Moyenne échantillon    : x̄ = {x_bar:.1f}")
print(f"Écart-type échantillon : s = {s:.1f}")
print(f"Erreur standard        : SE = {erreur_std:.1f}")
print(f"t critique (95%, df={n-1}) : {t_crit:.3f}")
print(f"IC 95% : [{ic_bas:.1f} ; {ic_haut:.1f}]")
print(f"Vraie moyenne population : {population_salaires.mean():.1f}")

# Méthode 2 : scipy
ic = stats.t.interval(confidence=0.95, df=n-1, loc=x_bar, scale=erreur_std)
print(f"\nIC 95% via scipy : {ic}")

## 6. Tests d'hypothèses

In [ ]:
# Logique des tests d'hypothèses :
# H0 : hypothèse nulle (statu quo)
# H1 : hypothèse alternative
# p-value : probabilité d'observer les données si H0 est vraie
# Si p-value < α (souvent 0.05) → rejeter H0

# Test t à un échantillon : la moyenne est-elle égale à 50000 ?
echantillon_it = np.random.normal(55000, 12000, 30)  # échantillon IT

t_stat, p_val = stats.ttest_1samp(echantillon_it, popmean=50000)
print("=== Test t (un échantillon) ===")
print(f"H0 : μ = 50000")
print(f"H1 : μ ≠ 50000")
print(f"t = {t_stat:.3f}, p-value = {p_val:.4f}")
if p_val < 0.05:
    print("→ p < 0.05 : on REJETTE H0 (différence significative)")
else:
    print("→ p ≥ 0.05 : on NE REJETTE PAS H0")

In [ ]:
# Test t à deux échantillons : comparer IT et Marketing
salaires_it = np.random.normal(58000, 13000, 40)
salaires_mkt = np.random.normal(52000, 10000, 40)

t_stat, p_val = stats.ttest_ind(salaires_it, salaires_mkt)
print("=== Test t (deux échantillons indépendants) ===")
print(f"IT Moyenne: {salaires_it.mean():.0f} | Mkt Moyenne: {salaires_mkt.mean():.0f}")
print(f"t = {t_stat:.3f}, p-value = {p_val:.4f}")
conclusion = "REJETTE H0" if p_val < 0.05 else "NE REJETTE PAS H0"
print(f"→ {conclusion}")

In [ ]:
# Test du chi-carré : indépendance entre deux variables catégorielles
# Ex : satisfaction liée au département ?
np.random.seed(1)
df_cat = pd.DataFrame({
    "dept": np.random.choice(["IT", "RH", "Finance"], 150),
    "satisfait": np.random.choice(["Oui", "Non"], 150)
})

table_contingence = pd.crosstab(df_cat["dept"], df_cat["satisfait"])
print("Table de contingence :")
print(table_contingence)

chi2, p_val, dof, expected = stats.chi2_contingency(table_contingence)
print(f"\nChi2 = {chi2:.3f}, ddl = {dof}, p-value = {p_val:.4f}")
print(f"→ {'DÉPENDANCE significative' if p_val < 0.05 else 'Pas de dépendance significative'}")

In [ ]:
# Corrélation de Pearson et Spearman
x_corr = np.random.normal(0, 1, 100)
y_corr = 2 * x_corr + np.random.normal(0, 1, 100)  # relation linéaire + bruit

r_pearson, p_pearson = stats.pearsonr(x_corr, y_corr)
r_spearman, p_spearman = stats.spearmanr(x_corr, y_corr)

print(f"Pearson  r = {r_pearson:.3f} (p={p_pearson:.4f}) → relations linéaires")
print(f"Spearman r = {r_spearman:.3f} (p={p_spearman:.4f}) → relations monotones, robuste aux outliers")

## 7. Bootstrap (simulation pour IC)

In [ ]:
# Bootstrap : rééchantillonnage avec remise pour estimer des IC
# Très utile quand on ne connaît pas la distribution

echantillon = np.random.choice(population_salaires, size=50, replace=False)
n_bootstrap = 10000

moyennes_boot = []
for _ in range(n_bootstrap):
    echantillon_boot = np.random.choice(echantillon, size=len(echantillon), replace=True)
    moyennes_boot.append(np.mean(echantillon_boot))

ic_bas_boot = np.percentile(moyennes_boot, 2.5)
ic_haut_boot = np.percentile(moyennes_boot, 97.5)

print(f"Bootstrap IC 95% : [{ic_bas_boot:.1f} ; {ic_haut_boot:.1f}]")
print(f"Vraie moyenne    : {population_salaires.mean():.1f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(moyennes_boot, bins=60, color="steelblue", alpha=0.8, edgecolor="white")
ax.axvline(ic_bas_boot, color="red", linestyle="--", label=f"IC 95% : [{ic_bas_boot:.0f} ; {ic_haut_boot:.0f}]")
ax.axvline(ic_haut_boot, color="red", linestyle="--")
ax.axvline(population_salaires.mean(), color="green", linestyle="-", label=f"Vraie moyenne : {population_salaires.mean():.0f}")
ax.set_title("Distribution Bootstrap des moyennes")
ax.legend()
plt.tight_layout()
plt.show()